d13_s3_a1.ipynb

In [ ]:
import numpy as np
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score


# Step 1: Generate Dataset
rng = np.random.default_rng(0)

y = rng.integers(0, 2, size=200)

scores = np.clip(
    0.5 + 0.3*(y*2-1) + rng.normal(0, 0.2, 200),
    0,
    1
)


# Step 2: Confusion Matrix at Given Threshold
def confusion_at(y_true, y_scores, threshold):

    y_true = np.asarray(y_true)
    y_scores = np.asarray(y_scores)

    y_pred = (y_scores >= threshold).astype(int)

    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))

    return {
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn
    }


# Step 3: Calculate Precision, Recall and F1
def precision_recall_f1(tp, fp, tn, fn):

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    return {
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1)
    }


# Step 4: Calculate ROC-AUC
def roc_auc(y_true, y_scores):

    y_true = np.asarray(y_true)
    y_scores = np.asarray(y_scores)

    n_positive = np.sum(y_true == 1)
    n_negative = np.sum(y_true == 0)

    if n_positive == 0 or n_negative == 0:
        return 0.0

    ranks = rankdata(y_scores)

    auc = (
        np.sum(ranks[y_true == 1])
        - n_positive * (n_positive + 1) / 2
    ) / (n_positive * n_negative)

    return float(auc)


# Step 5: Find Best Threshold Based on F1
def best_threshold(y_true, y_scores, thresholds):

    best_t = None
    best_f1 = -1.0

    for threshold in thresholds:

        c = confusion_at(y_true, y_scores, threshold)

        metrics = precision_recall_f1(
            c["tp"],
            c["fp"],
            c["tn"],
            c["fn"]
        )

        if metrics["f1"] > best_f1:
            best_f1 = metrics["f1"]
            best_t = threshold

    return best_t


# Step 6: Verify Using Given Example
y_test = np.array([0, 0, 1, 1, 1])

s_test = np.array([0.2, 0.6, 0.4, 0.7, 0.9])

print("Confusion Matrix:")
c = confusion_at(y_test, s_test, 0.5)
print(c)


# Step 7: Precision, Recall and F1
print("\nPrecision, Recall and F1:")

metrics = precision_recall_f1(
    c["tp"],
    c["fp"],
    c["tn"],
    c["fn"]
)

print(metrics)


# Step 8: ROC-AUC
print("\nROC-AUC:")
print(round(roc_auc(y_test, s_test), 3))


# Step 9: Best Threshold
print("\nBest Threshold:")

threshold = best_threshold(
    y_test,
    s_test,
    [0.3, 0.5, 0.7]
)

print(threshold)


# Step 10: Test Zero-Division Case
print("\nZero-Division Test:")

print(precision_recall_f1(0, 0, 5, 3))


# Step 11: Test Perfect Separation
print("\nPerfect Separation AUC:")

y_perfect = np.array([0, 0, 1, 1])
s_perfect = np.array([0.1, 0.2, 0.8, 0.9])

print(roc_auc(y_perfect, s_perfect))


# Step 12: Verify AUC with sklearn
print("\nSklearn ROC-AUC Verification:")

print(roc_auc_score(y_test, s_test))